In [ ]:
import strangeworks 
from strangeworks.qiskit import StrangeworksProvider


# Authenticate to strangeworks first
strangeworks.authenticate(username="{your-username}", api_key="{your-api-key}")

sw_provider = StrangeworksProvider()
print(sw_provider.backends())

In [ ]:
# run a single random circuit
from qiskit.circuit.random.utils import random_circuit

qc = random_circuit(num_qubits=7, depth=10, measure=True)

backend = sw_provider.get_backend("rigetti.rigetti-managed.Aspen-M-1")
job = backend.run(qc, shots=10)
results = job.result()

print(f"results: {results}")

In [ ]:
# finally QAOA!
import networkx as nx
import numpy as np

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer

from qiskit.utils import algorithm_globals, QuantumInstance
from qiskit import BasicAer
from qiskit.algorithms import QAOA
from qiskit.algorithms.optimizers import SPSA

import strangeworks
from strangeworks.qiskit import StrangeworksProvider

seed = 1234
algorithm_globals.random_seed = seed

# Generate a graph of 4 nodes
n = 4
graph = nx.Graph()
graph.add_nodes_from(np.arange(0, n, 1))
elist = [(0, 1, 1.0), (0, 2, 1.0), (0, 3, 1.0), (1, 2, 1.0), (2, 3, 1.0)]
graph.add_weighted_edges_from(elist)

# Compute the weight matrix from the graph
w = nx.adjacency_matrix(graph)

# Formulate the problem as quadratic program
problem = QuadraticProgram()
_ = [problem.binary_var(f"x{i}") for i in range(n)]  # create n binary variables
linear = w.dot(np.ones(n))
quadratic = -w
problem.maximize(linear=linear, quadratic=quadratic)

# Fix node 0 to be 1 to break the symmetry of the max-cut solution
problem.linear_constraint([1, 0, 0, 0], '==', 1)

spsa = SPSA(maxiter=250)

backend = sw_provider.get_backend("rigetti.rigetti-managed.Aspen-M-1")
q_i = QuantumInstance(backend=backend, seed_simulator=seed, seed_transpiler=seed)
qaoa = QAOA(optimizer=spsa, reps=5, quantum_instance=q_i)
algorithm = MinimumEigenOptimizer(qaoa)
result = algorithm.solve(problem)
print(result)